# Week 6 Assessment — MCP-Powered Portfolio Research Agent

**Author: Jemine Mene-Ejegi**

Enhanced version of the Week 6 MCP labs :

| Enhancement | What it does | Source requirement |
|---|---|---|
| **Custom MCP Server** | `portfolio_notes_server.py` — fully local SQLite server exposing tools AND a `notes://{symbol}` MCP resource | *"Make your own MCP Server"* — lab 2 exercise |
| **All 3 MCP Architectures** | Type 1 (local-only notes), Type 2 (Tavily web search), Type 3 (date server via HTTP/SSE) | *"using all 3 approaches"* — lab 3 exercise |
| **MCP Resources** | Prior research notes exposed as `notes://{symbol}` resources, readable by agents as context | MCP resources pattern from `accounts_server.py` |
| **Multi-Agent Research Pipeline** | Researcher (web) → Notes Manager (local) → Portfolio Advisor (orchestrator across all 3 types) | *"Add some more agents to the mix"* — captions.txt |
| **Native OpenAI MCP Client** | `portfolio_client.py` calls the notes server and reads resources without the Agents SDK | Harder exercise from lab 2 |
| **Gradio UI** | Interactive stock research interface with ticker input and chat history | Consistent with weeks 2 & 4 pattern |

In [ ]:
import asyncio
import json
import os
import subprocess
import sys
from contextlib import AsyncExitStack

import openai
import gradio as gr
from agents import Agent, FunctionTool, Runner, gen_trace_id, trace
from agents.mcp import MCPServerSse, MCPServerStdio
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Navigate to the 6_mcp root so that project modules and uv run paths resolve correctly
cwd = os.getcwd()
if "community_contributions" in cwd or cwd.endswith("jaymineh"):
    root = os.path.abspath(os.path.join(cwd, "../.."))
    os.chdir(root)
    sys.path.insert(0, root)

print(f"Working directory: {os.getcwd()}")

In [ ]:
load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

print(f"OpenAI key : {'found (' + openai_key[:8] + '...)' if openai_key else 'MISSING'}")
print(f"Tavily key : {'found' if tavily_key else 'MISSING — researcher agent will not search the web'}")

## Architecture

```
User enters stock ticker + research question
              │
              ▼
    Researcher Agent  ─── Type 2: Tavily MCP (local subprocess → web API)
      └─ searches for news, earnings, analyst ratings
              │
              ▼  (as tool)
    Notes Manager Agent ── Type 1: portfolio_notes_server (fully local SQLite)
      ├─ tool: save_research_note    ← persists findings to DB
      ├─ tool: get_research_notes    ← retrieves prior research
      └─ tool: list_tracked_symbols  ← shows coverage
              │
              ▼  (as tool)
    Portfolio Advisor Agent (orchestrator)
      ├─ tool: researcher_tool
      ├─ tool: notes_manager_tool
      └─ MCPServerSse → date_server via HTTP/SSE ── Type 3: remote-style
              │
    ┌─────────┴──────────────────┐
    ▼                            ▼
  Agents SDK pipeline      Native OpenAI client
  (run_stock_research)     (portfolio_client.py)
                                 └─ reads notes://{symbol} MCP resource
              │
              ▼
          Gradio UI
```

---
## 1 — Custom MCP Server: `portfolio_notes_server.py` (Type 1 — fully local)

A home-made FastMCP server (inspired by `accounts_server.py`) that persists stock research
notes to a local SQLite database and exposes them both as **tools** and as a
`notes://{symbol}` **MCP resource** — so agents can read prior research as structured context.

In [ ]:
NOTES_SERVER_PATH = "community_contributions/jaymineh/portfolio_notes_server.py"
notes_params = {"command": "uv", "args": ["run", NOTES_SERVER_PATH]}

async with MCPServerStdio(params=notes_params, client_session_timeout_seconds=30) as server:
    notes_tools = await server.list_tools()

print("Tools exposed by portfolio_notes_server:")
for t in notes_tools:
    print(f"  {t.name}: {t.description.strip().splitlines()[0]}")

print("\nMCP Resource exposed:")
print("  notes://{symbol}  — read all research notes for a ticker as a resource")

---
## 2 — Researcher Agent using Tavily (Type 2 — local subprocess calling web API)

The Researcher Agent uses the `tavily-mcp` npm package to search the web for news
and analysis about a given stock.  It is exposed as a **tool** so the Portfolio Advisor
can invoke it autonomously, exactly like the Researcher in `traders.py`.

In [ ]:
tavily_params = {
    "command": "npx",
    "args": ["-y", "tavily-mcp"],
    "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")},
}

# Verify Tavily tools are available
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=30) as server:
    tavily_tools = await server.list_tools()

print("Tools exposed by tavily-mcp:")
for t in tavily_tools:
    print(f"  {t.name}")

---
## 3 — Notes Manager Agent (Type 1 — fully local)

The Notes Manager Agent uses `portfolio_notes_server.py` to save and retrieve research notes.
After the Researcher finds news, the Portfolio Advisor calls this agent to persist key findings
so they can be retrieved as context in future research sessions.

In [ ]:
def make_researcher_agent(tavily_server) -> Agent:
    return Agent(
        name="Researcher",
        instructions=(
            "You are a financial researcher. Search the web for the latest news, earnings updates, "
            "analyst ratings, and key events for the stock you are asked about. "
            "Produce a comprehensive but concise summary (3-5 paragraphs) of your findings."
        ),
        model="gpt-5.4-mini",
        mcp_servers=[tavily_server],
    )


def make_notes_manager_agent(notes_server) -> Agent:
    return Agent(
        name="Notes Manager",
        instructions=(
            "You manage a persistent research notes database. "
            "When asked to save a note, call save_research_note with the ticker and the note text. "
            "When asked to retrieve notes, call get_research_notes with the ticker. "
            "When asked for an overview, call list_tracked_symbols."
        ),
        model="gpt-5.4-mini",
        mcp_servers=[notes_server],
    )

print("Agent factory functions ready.")

---
## 4 — Portfolio Advisor + Type 3 (remote SSE) date context

The Portfolio Advisor is the orchestrating agent.  It has access to:
- The **Researcher** and **Notes Manager** as tools (agents-as-tools pattern)
- The `date_server.py` we built in lab 3 — connected via **`MCPServerSse`** (HTTP/SSE, Type 3)
  to get current-date context for its analysis

The SSE server is started as a background subprocess before connecting, mirroring exactly how
a real hosted remote MCP server would be used.

In [ ]:
ADVISOR_INSTRUCTIONS = """
You are a portfolio research advisor. Your job is to research a stock and produce a
clear, actionable investment perspective.

YOUR PROCESS:
1. Call research_stock to get the latest web news and analyst sentiment.
2. Call retrieve_prior_notes to check if we have prior research saved for this ticker.
3. Synthesise the web research with any prior notes and the current date context.
4. Call save_note to persist 2-3 key bullet-point insights for future reference.
5. Respond with a structured investment perspective covering:
   - Current market sentiment and recent news
   - Key risks and opportunities
   - Overall outlook (bullish / neutral / bearish) with reasoning
   - Any prior research context that is still relevant

Be analytical and concise. Do not ask for confirmation — proceed autonomously.
"""

print("Portfolio Advisor instructions ready.")

---
## 5 — Full Research Pipeline

`run_stock_research()` wires together all 3 MCP server types in a single
`AsyncExitStack`, starts the SSE server as a background process, builds the
agent hierarchy, and runs the Portfolio Advisor end-to-end.

In [ ]:
async def _wait_for_port(port: int, timeout: float = 15.0):
    """Poll localhost:port until it accepts a TCP connection or timeout is reached."""
    import socket
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        try:
            with socket.create_connection(("localhost", port), timeout=0.5):
                return  # port is open
        except OSError:
            await asyncio.sleep(0.5)
    raise TimeoutError(f"SSE server did not start within {timeout}s")


async def run_stock_research(ticker: str, question: str) -> tuple[str, str]:
    """
    Full multi-agent pipeline using all 3 MCP server types.
    Returns (final_output, trace_url).
    """
    ticker = ticker.strip().upper()

    # Start date_server.py in SSE mode (Type 3 — remote-style HTTP server)
    sse_proc = subprocess.Popen(["uv", "run", "date_server.py", "--sse"])
    await _wait_for_port(8000)  # wait until the server is actually ready

    try:
        async with AsyncExitStack() as stack:
            # TYPE 1: fully local notes server
            notes_server = await stack.enter_async_context(
                MCPServerStdio(params=notes_params, client_session_timeout_seconds=60)
            )
            # TYPE 2: local subprocess calling Tavily web API
            tavily_server = await stack.enter_async_context(
                MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60)
            )
            # TYPE 3: remote-style HTTP/SSE server
            date_server = await stack.enter_async_context(
                MCPServerSse(
                    params={"url": "http://localhost:8000/sse"},
                    client_session_timeout_seconds=30,
                )
            )

            # Build sub-agents and expose as tools
            researcher = make_researcher_agent(tavily_server)
            notes_manager = make_notes_manager_agent(notes_server)

            research_tool = researcher.as_tool(
                tool_name="research_stock",
                tool_description=(
                    f"Search the web for the latest news, earnings, and analyst sentiment "
                    f"about {ticker}. Returns a comprehensive summary."
                ),
            )
            notes_retrieve_tool = notes_manager.as_tool(
                tool_name="retrieve_prior_notes",
                tool_description=(
                    f"Retrieve any previously saved research notes for {ticker} from the "
                    f"local database."
                ),
            )
            notes_save_tool = notes_manager.as_tool(
                tool_name="save_note",
                tool_description=(
                    f"Save a key research insight or finding for {ticker} to the local "
                    f"database for future reference."
                ),
            )

            # Portfolio Advisor: all 3 types in one agent
            advisor = Agent(
                name="Portfolio Advisor",
                instructions=ADVISOR_INSTRUCTIONS,
                model="gpt-5.4-mini",
                tools=[research_tool, notes_retrieve_tool, notes_save_tool],
                mcp_servers=[date_server],  # Type 3: date context via SSE
            )

            trace_id = gen_trace_id()
            trace_url = f"https://platform.openai.com/traces/trace?trace_id={trace_id}"

            with trace("Portfolio Research", trace_id=trace_id):
                result = await Runner.run(
                    advisor,
                    f"Research {ticker}. Question: {question}",
                    max_turns=25,
                )

        return result.final_output, trace_url

    finally:
        sse_proc.terminate()


print("run_stock_research() ready.")

In [ ]:
# Quick smoke test — try NVDA
result, trace_url = await run_stock_research("NVDA", "What is the current market sentiment and outlook?")
print(f"Trace: {trace_url}\n")
display(Markdown(result))

---
## 6 — Native OpenAI MCP Client (no Agents SDK)

`portfolio_client.py` implements the harder lab 2 exercise for the notes server:
connect to `portfolio_notes_server.py` via stdio, convert its tools to the native
OpenAI function-calling format, and drive a tool-use loop with `openai.AsyncOpenAI`
directly — zero Agents SDK involved.

It also reads the `notes://{symbol}` **MCP resource** to show how resources flow
into the conversation as context.

In [ ]:
from community_contributions.jaymineh.portfolio_client import (
    call_notes_tool,
    get_notes_tools_openai_format,
    read_notes_resource,
)

# Get tools in native OpenAI format
openai_notes_tools = await get_notes_tools_openai_format()
print("Notes tools in native OpenAI format:")
for t in openai_notes_tools:
    print(f"  {t['function']['name']}: {t['function']['description'].strip().splitlines()[0]}")

In [ ]:
# Read the notes resource for NVDA (populated by the pipeline above)
nvda_notes_resource = await read_notes_resource("NVDA")
print("MCP Resource — notes://NVDA:")
print(nvda_notes_resource)

# Now drive a native OpenAI tool-use loop — no Agents SDK
oai_client = openai.AsyncOpenAI()
messages = [
    {"role": "system", "content": "You are a portfolio assistant with access to a research notes database."},
    {"role": "user", "content": f"Here is prior research context:\n{nvda_notes_resource}\n\nPlease list all symbols we are tracking."},
]

response = await oai_client.chat.completions.create(
    model="gpt-5.4-mini",
    tools=openai_notes_tools,
    messages=messages,
)

while response.choices[0].finish_reason == "tool_calls":
    assistant_msg = response.choices[0].message
    messages.append(assistant_msg)
    tool_results = []
    for call in assistant_msg.tool_calls:
        args = json.loads(call.function.arguments)
        print(f"  Calling MCP tool '{call.function.name}' → {args}")
        result = await call_notes_tool(call.function.name, args)
        print(f"  Result: {result[:120]}...")
        tool_results.append({"role": "tool", "tool_call_id": call.id, "content": result})
    messages.extend(tool_results)
    response = await oai_client.chat.completions.create(
        model="gpt-5.4-mini", tools=openai_notes_tools, messages=messages
    )

display(Markdown(response.choices[0].message.content))

---
## 7 — Gradio UI

Interactive chat interface where you enter a stock ticker and ask any research question.
The full multi-agent pipeline (all 3 MCP types) runs on each message and streams the
Portfolio Advisor's structured investment perspective back to the chat.

In [ ]:
async def research_chat(message: str, history: list, ticker: str) -> str:
    ticker = ticker.strip().upper()
    if not ticker:
        return "Please enter a stock ticker symbol in the field above (e.g. AAPL, TSLA, NVDA)."
    try:
        result, trace_url = await run_stock_research(ticker, message)
        return f"{result}\n\n---\n*[View OpenAI trace]({trace_url})*"
    except Exception as e:
        return f"Research failed: {e}"


with gr.Blocks(theme=gr.themes.Soft(primary_hue="sky")) as ui:
    gr.Markdown("## Portfolio Research Agent — Week 6 MCP Assessment")
    gr.Markdown(
        "Enter a stock ticker and ask any research question. The agent will:\n"
        "1. **Search the web** for the latest news (Tavily MCP — Type 2)\n"
        "2. **Retrieve prior notes** from the local database (portfolio_notes_server — Type 1)\n"
        "3. **Get current date context** via HTTP/SSE (date_server — Type 3)\n"
        "4. **Save key insights** to the notes database for future sessions\n\n"
        "⚠️ Each run makes web searches (~$0.01–$0.05 in API costs)."
    )

    ticker_input = gr.Textbox(
        label="Stock Ticker",
        placeholder="e.g. AAPL",
        scale=1,
        max_lines=1,
    )

    chat = gr.ChatInterface(
        fn=research_chat,
        additional_inputs=[ticker_input],
        type="messages",
        examples=[
            ["What is the current market sentiment and investment outlook?"],
            ["Summarise the latest earnings and analyst ratings."],
            ["What are the key risks I should be aware of right now?"],
        ],
    )

print("Gradio UI ready.")

In [ ]:
ui.launch()